In [ ]:
from pathlib import Path
import sys
import os
import clingo
project_root = "."


In [ ]:
from typing import Any

INSTANCE = "tree"

ASPGARP_FILES = [
    f"{project_root}/single/sim.lp",
    f"{project_root}/single/cnst.lp",
    f"{project_root}/adapter.lp",
]

REFERENCE_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp",
    f"{project_root}/models/{INSTANCE}/ref.lp",
]

BASIC_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp"
]

META_FILES = [
    f"{project_root}/meta.lp",
]

OUTPUT = f"{project_root}/out/{INSTANCE}"

#create output directory if it doesn't exist
os.makedirs(OUTPUT, exist_ok=True)

In [ ]:
import clingo


def states_from_reference_model(
    RMODEL: list[str], ASPGARPFILES: list[str]
) -> list[dict[str, tuple[str, str]]]:
    states: list[dict[str, tuple[str, str]]] = []

    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in RMODEL + ASPGARPFILES:
        ctl.load(path)

    ctl.add("base", [], "#show obs/1.")
    ctl.add("base", [], ":- violation.")
    ctl.ground([("base", [])])

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            state: dict[str, tuple[str, str]] = {}
            for atom in model.symbols(shown=True):
                if atom.name == "obs" and len(atom.arguments) == 1:
                    varname, value, direction = atom.arguments[0].arguments
                    state[varname.name] = (value.name, direction.name)
            states.append(state)

    return states


def states_to_description(
    states: list[dict[str, tuple[str, str]]]
) -> list[str]:
    descriptions: list[str] = []

    for i, state in enumerate(states):
        parts = [f"holds(SID,{varname},{value},{direction})"
                 for varname, (value, direction) in state.items()]
        description = f"state(SID,{i}) :- " + ", ".join(parts) + "."
        descriptions.append(description)

    return descriptions

def states_to_labels(states: list[dict[str, tuple[str, str]]]) -> list[str]:
    labels: list[str] = []

    for i, state in enumerate(states):
        parts = [f"{varname}=({value},{direction})"
                 for varname, (value, direction) in state.items()]
        label = f"statelabel({i},\"" + ",\\n ".join(parts) + "\")."
        labels.append(label)

    return labels


In [ ]:
# Reuse the above and find all the transitions between them

def transitions_from_reference_model(RMODEL, ASPGARPFILES, state_descriptions) -> list[str]:
    transitions = []
    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in REFERENCE_MODEL + ASPGARP_FILES:
        ctl.load(path)
    ctl.add("base", [],  "\n".join(state_descriptions))
    ctl.add("base", [], ":- violation.")
    ctl.add("base", [], "#show state/2.")
    ctl.ground([("base", [])])

    for i, m in enumerate(ctl.solve(yield_=True)):
        from_state = None
        to_state = None
        for atom in m.symbols(shown=True):
            if atom.name == "state" and len(atom.arguments) == 2:
                if atom.arguments[0] == clingo.Number(0): 
                    from_state = atom
                else:
                    to_state = atom
        if from_state and to_state:
            transitions.append(f"edge(({from_state.arguments[1]},{to_state.arguments[1]})).")
    return transitions

# Step 0: Generate Positive Examples

In [ ]:

states = states_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES)
state_descriptions = states_to_description(states)
state_labels = states_to_labels(states)
transition_descriptions = transitions_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES, state_descriptions)

#write the state and transition descriptions to files
with open(f"{OUTPUT}/transitions.lp", "w") as f:
    f.write("\n".join(transition_descriptions))
    f.write("\n".join(state_labels))

with open(f"{OUTPUT}/states.lp", "w") as f:
    f.write("\n".join(state_descriptions))
    

!clingo --project --warn=none --outf=2 {OUTPUT}/transitions.lp  | clingraph --out=render --format=svg --viz-encoding=viz.lp

# Step 1: 

## Pruning



In [ ]:
states

def state_to_assumptions(state) -> list[tuple[clingo.Symbol, bool]]:
    assumptions: list[tuple[clingo.Symbol, bool]] = []
    for varname, (value, direction) in state.items():
        atom = clingo.Function(
            "holds",
            [
                clingo.Function(varname),
                clingo.Function(value),
                clingo.Function(direction),
            ],
        )
        wrapper = clingo.Function("obs", [atom])
        assumptions.append((wrapper, True))
    return assumptions
print(state_to_assumptions(states[0]))

In [ ]:
def find_hypotheses(
    pos_states,
    aspqsim_files: list[str],
    model_files: list[str],
    meta_files: list[str],
    *,
    verbose: bool = True,
    options = [],
    hypothesis_constraints = []
) -> list[list[tuple[clingo.Symbol, bool]]]:
    """
    Search hypotheses for positive examples and accumulate nogoods
    to prune previously accepted solutions.
    """
    files = aspqsim_files + model_files + meta_files

    ctl = clingo.Control(["0", "--project", "--warn=none"] + options)
    for path in files:
        ctl.load(path)

    ctl.add("base", [], "\n".join(hypothesis_constraints))
    ctl.add("base", [], "#show abduced/1.")
    ctl.ground([("base", [])])

    violation = clingo.parse_term("violation")
    hypothesis_constraints: list[list[tuple[clingo.Symbol, bool]]] = []

    for key, pos_example in enumerate(pos_states):
        if verbose:
            print(f"Example ID: {key}")

        base_assumptions = state_to_assumptions(pos_example)

        if verbose:
            for atom, _ in base_assumptions:
                print(f"Adding assumption: {atom}")

        solve_assumptions =  base_assumptions + [(violation, True)]
        found_valid_model = False
        added_existing_nogoods = False

        with ctl.solve(yield_=True, assumptions=solve_assumptions) as handle:
            counter = 0
            for model in handle:
                if verbose:
                    print("Model found, checking for violation...")

                if not added_existing_nogoods:
                    for nogood in hypothesis_constraints:
                        model.context.add_nogood(nogood)
                    added_existing_nogoods = True

                atoms = model.symbols(shown=True)

                # Skip models already ruled out by an existing nogood.
                if any(all(atom in atoms for atom, _ in nogood) for nogood in hypothesis_constraints):
                    if verbose:
                        print("Fluke detected, skipping this model.")
                    continue

                found_valid_model = True

                if verbose:
                    print(atoms)

                nogood = [(atom, True) for atom in atoms]
                hypothesis_constraints.append(nogood)
                model.context.add_nogood(nogood)

        if not found_valid_model and verbose:
            print(
                "No model found for this example, even without the nogood constraints. "
            )
    print(counter)
    return hypothesis_constraints

In [ ]:
hypothesis_constraints = find_hypotheses(states, ASPGARP_FILES, BASIC_MODEL, META_FILES, verbose=True)

constraints = []

for nogood in hypothesis_constraints:
    constraints.append(":- " + ", ".join(f"{atom}".replace("rl(1)","RULE1").replace("rl(2)","RULE2") for atom, val in nogood) + ".")

print(len(constraints), "constraints generated from nogoods.")

constraints
    

# Step 2:

## Justification Assignments